In [3]:
pip install holidays

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.3 MB 11.9 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Final_data_feature_engineering
# Input: dataset_mansion_features.csv
# Output: dataset_mansion_features_FINAL.csv
import pandas as pd
import holidays

print("Loading raw dataset...")
df = pd.read_csv("dataset_mansion_features.csv")

# ------------------------------------------
# 1. BASIC CLEANING
# ------------------------------------------

# Drop extra index column
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Fix datetime
df["MTime"] = pd.to_datetime(df["MTime"])


# ------------------------------------------
# 2. ADD CONTEXT FEATURES
# ------------------------------------------

# Year
df["year"] = df["MTime"].dt.year

# Season
def get_season(month):
    if month in [12, 1, 2]:
        return 0  # Winter
    if month in [3, 4, 5]:
        return 1  # Spring
    if month in [6, 7, 8]:
        return 2  # Summer
    return 3      # Autumn

df["season"] = df["month"].apply(get_season)

# Extreme cold indicator
df["very_cold"] = (df["AirTemperature(degC)"] < -5).astype(int)

# Interaction term
df["humidity_temp_interaction"] = (
    df["RelativeHumidity(%)"] * df["AirTemperature(degC)"]
)

# House structural change
df["has_heat_pump"] = (df["MTime"] >= pd.Timestamp("2021-10-01")).astype(int)


# ------------------------------------------
# 3. ADD 48h & 72h LAG FEATURES
# ------------------------------------------

for lag in [48, 72]:
    df[f"cons_lag_{lag}"] = df["Consumption"].shift(lag)


# ------------------------------------------
# 4. ADD ROLLING WINDOWS (mean + std)
# ------------------------------------------

for window in [48, 72]:
    df[f"cons_roll_mean_{window}"] = df["Consumption"].rolling(window).mean()
    df[f"cons_roll_std_{window}"] = df["Consumption"].rolling(window).std()


# ------------------------------------------
# 5. DROP NaNs FROM LAGS + ROLLINGS
# ------------------------------------------

before_rows = len(df)
df = df.dropna().reset_index(drop=True)
after_rows = len(df)

print(f"Dropped {before_rows - after_rows} rows due to lag/rolling NaNs.")


# ------------------------------------------
# 6. ADD HOLIDAY FEATURE
# ------------------------------------------

print("Adding holiday feature...")

# Finland holidays for full year range
years = range(df["MTime"].dt.year.min(), df["MTime"].dt.year.max() + 1)
fi_holidays = holidays.Finland(years=years)

holiday_dates = set(fi_holidays.keys())

# Extract date only
df["date_only"] = df["MTime"].dt.date

df["is_holiday"] = df["date_only"].isin(holiday_dates).astype(int)

# Remove helper column
df = df.drop(columns=["date_only"])


# ------------------------------------------
# 7. SAVE FINAL DATASET
# ------------------------------------------

df.to_csv("dataset_mansion_features_FINAL.csv", index=False)

print("SUCCESS: dataset_mansion_features_FINAL.csv created.")
print(f"Final rows: {len(df)}")


Loading raw dataset...
Dropped 169 rows due to lag/rolling NaNs.
Adding holiday feature...
SUCCESS: dataset_mansion_features_FINAL.csv created.
Final rows: 56062
